# RadiNova AI — Limb Fracture Model Training (DenseNet-121 + Grad-CAM)

**Objective:** Train a high-sensitivity DenseNet-121 binary classifier for Bone / Limb Fracture detection on the Kaggle `devbatrax/fracture-detection-using-x-ray-images` dataset with 80/10/10 stratified split, evaluate clinical metrics (Recall / Sensitivity as priority), verify Grad-CAM heatmap localization, and export `limb_densenet121.pth` weights for the RadiNova AI clinical platform.

> **Clinical Safety Disclaimer:** *For educational and investigational research purposes only. Not certified as a primary standalone diagnostic medical device.*

### Step 1: Install Dependencies & Setup Environment

In [ ]:
!pip install -q kagglehub torch torchvision scikit-learn pandas pillow opencv-python matplotlib
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Step 2: Download Limb Fracture Dataset & Perform 80/10/10 Stratified Re-Split

In [ ]:
import kagglehub
import os, random, csv
from pathlib import Path
import pandas as pd

print("Downloading Kaggle limb fracture dataset (devbatrax/fracture-detection-using-x-ray-images)...")
dataset_path = kagglehub.dataset_download("devbatrax/fracture-detection-using-x-ray-images")
print(f"Downloaded to: {dataset_path}")

valid_exts = {".jpeg", ".jpg", ".png", ".JPEG", ".JPG", ".PNG"}
samples = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if Path(f).suffix in valid_exts:
            full_p = os.path.join(root, f)
            p_up = full_p.upper()
            if "NOT FRACTURED" in p_up or "NOT_FRACTURED" in p_up or "NORMAL" in p_up:
                samples.append((full_p, "NOT_FRACTURED"))
            elif "FRACTURED" in p_up or "FRACTURE" in p_up:
                samples.append((full_p, "FRACTURED"))

print(f"Total images scanned: {len(samples)}")

buckets = {}
for p, l in samples:
    buckets.setdefault(l, []).append((p, l))

rng = random.Random(42)
manifest = []
for lbl, items in buckets.items():
    rng.shuffle(items)
    n = len(items)
    n_tr = int(n * 0.80)
    n_v = int(n * 0.10)
    for p, _ in items[:n_tr]: manifest.append({"filepath": p, "label": lbl, "split": "train"})
    for p, _ in items[n_tr:n_tr+n_v]: manifest.append({"filepath": p, "label": lbl, "split": "val"})
    for p, _ in items[n_tr+n_v:]: manifest.append({"filepath": p, "label": lbl, "split": "test"})

df_manifest = pd.DataFrame(manifest)
df_manifest.to_csv("limb_manifest.csv", index=False)
print("Manifest saved: limb_manifest.csv")
print(df_manifest.groupby(["split", "label"]).size())

### Step 3: PyTorch Dataset & DenseNet-121 Architecture Setup

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CLASS_TO_IDX = {"NOT_FRACTURED": 0, "FRACTURED": 1}
CLASSES = ["NOT_FRACTURED", "FRACTURED"]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

class LimbDataset(Dataset):
    def __init__(self, df, split, tf):
        self.data = df[df["split"] == split].reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        return self.tf(img), CLASS_TO_IDX[row["label"]]

train_ds = LimbDataset(df_manifest, "train", train_tf)
val_ds = LimbDataset(df_manifest, "val", val_tf)
test_ds = LimbDataset(df_manifest, "test", val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

# Build DenseNet-121
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, 2)
)
model = model.to(device)
print("DenseNet-121 initialized with binary classifier head for Fracture detection.")

### Step 4: Model Training with Cosine Annealing LR & Best F1 Tracking

In [ ]:
def eval_metrics(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model(imgs)
            _, preds = torch.max(outs, 1)
            all_p.extend(preds.cpu().numpy())
            all_l.extend(lbls.cpu().numpy())
    y_true, y_pred = np.array(all_l), np.array(all_p)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    acc = (tp + tn) / max(len(y_true), 1)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * (prec * rec) / max(prec + rec, 1e-6)
    return acc, prec, rec, f1, (tn, fp, fn, tp)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

best_f1 = 0.0
epochs = 10

for epoch in range(1, epochs + 1):
    model.train()
    r_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        r_loss += loss.item() * imgs.size(0)
    scheduler.step()
    
    val_acc, val_prec, val_rec, val_f1, _ = eval_metrics(model, val_loader)
    print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {r_loss/len(train_ds):.4f} | Val Acc: {val_acc*100:.1f}% Rec (Sens): {val_rec*100:.1f}% F1: {val_f1:.4f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": CLASSES,
            "val_metrics": {"accuracy": val_acc, "precision": val_prec, "recall_sensitivity": val_rec, "f1_score": val_f1},
            "epoch": epoch
        }, "limb_densenet121.pth")
        print(f"  >>> Checkpoint saved (F1: {best_f1:.4f}) -> limb_densenet121.pth")

### Step 5: Test Set Performance & Clinical Metric Evaluation

In [ ]:
checkpoint = torch.load("limb_densenet121.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
acc, prec, rec, f1, (tn, fp, fn, tp) = eval_metrics(model, test_loader)

print("="*55)
print("   RADINOVA AI — LIMB FRACTURE TEST EVALUATION")
print("="*55)
print(f"Accuracy:                 {acc*100:.2f}%")
print(f"Precision (PPV):          {prec*100:.2f}%")
print(f"Recall / Sensitivity:     {rec*100:.2f}%  <-- CLINICAL PRIORITY (MINIMIZE MISSES)")
print(f"Specificity (TNR):        {tn/(tn+fp)*100:.2f}%")
print(f"F1-Score:                 {f1:.4f}")
print("-"*55)
print(f"Confusion Matrix:")
print(f"                     Pred NOT_FRACTURED    Pred FRACTURED")
print(f"Actual NOT_FRACTURED {tn:<22} {fp:<22}")
print(f"Actual FRACTURED     {fn:<22} {tp:<22}")
print("="*55)

### Step 6: Grad-CAM Explainability Heatmap Verification

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Select a sample fractured radiograph from the test set
test_fractures = df_manifest[(df_manifest["split"] == "test") & (df_manifest["label"] == "FRACTURED")]
sample_path = test_fractures.iloc[0]["filepath"]

raw_img = Image.open(sample_path).convert("RGB")
inp_tensor = val_tf(raw_img).unsqueeze(0).to(device)

# Hook final convolutional layer
target_layer = model.features[-1]
activations = []
gradients = []

def f_hook(mod, inp, out): activations.append(out)
def b_hook(mod, grad_in, grad_out): gradients.append(grad_out[0])

h1 = target_layer.register_forward_hook(f_hook)
h2 = target_layer.register_full_backward_hook(b_hook)

model.eval()
out = model(inp_tensor)
prob = torch.softmax(out, dim=-1)[0, 1].item()
model.zero_grad()
out[0, 1].backward()

h1.remove()
h2.remove()

grad = gradients[0].cpu().data.numpy()[0]
act = activations[0].cpu().data.numpy()[0]
weights = np.mean(grad, axis=(1, 2))
cam = np.zeros(act.shape[1:], dtype=np.float32)
for i, w in enumerate(weights):
    cam += w * act[i, :, :]
cam = np.maximum(cam, 0)
cam = cv2.resize(cam, (224, 224))
cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
orig_resized = np.array(raw_img.resize((224, 224)))
overlay = np.uint8(0.6 * orig_resized + 0.4 * heatmap)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(orig_resized); ax[0].set_title("Original Radiograph"); ax[0].axis('off')
ax[1].imshow(heatmap); ax[1].set_title("Grad-CAM Heatmap"); ax[1].axis('off')
ax[2].imshow(overlay); ax[2].set_title(f"Overlay (Fracture Prob: {prob*100:.1f}%)"); ax[2].axis('off')
plt.tight_layout()
plt.show()

### Step 7: Export Trained Weights (`limb_densenet121.pth`)

In [ ]:
from google.colab import files
files.download("limb_densenet121.pth")
print("Downloaded! Place 'limb_densenet121.pth' in 'model/weights/limb_densenet121.pth' inside RadiNova AI.")